In [1]:
#Ejemplos

In [2]:
import numpy as np
from numpy.linalg import eig
from scipy.optimize import linprog, minimize

# =========================================================
# Configuration
# =========================================================

SEED = 42
N_EXAMPLE = 5
MAX_TRIES = 80000
ALPHA_VALUES = list(range(2, 10))
BASELINE_ALPHA = 2
RANK_TOL = 1e-10

# Bounds for optimisation-based methods
LOG_WEIGHT_BOUND = 8.0
OPT_MAXITER = 300

np.random.seed(SEED)


# =========================================================
# Utility functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.any(~np.isfinite(w)):
        raise ValueError("Priority vector contains non-finite values.")

    if np.any(w < 0):
        raise ValueError("Priority vector contains negative values.")

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def geometric_mean_start(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    gm = normalize_positive(gm)
    return np.log(gm)


def log_variables_to_weights(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    z = z - np.max(z)
    w = np.exp(z)
    return w / np.sum(w)


def fixed_scale_log_vector(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    return np.concatenate([z, np.array([0.0])])


def ratio_matrix_from_log_vector(x):
    x = np.asarray(x, dtype=float)
    d = x[:, None] - x[None, :]
    d = np.clip(d, -2 * LOG_WEIGHT_BOUND, 2 * LOG_WEIGHT_BOUND)
    return np.exp(d)


# =========================================================
# Priority derivation methods
# =========================================================

def eigenvector_priority(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    w = eigenvectors[:, idx].real

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    return normalize_positive(w)


def geometric_mean_priority(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    return normalize_positive(gm)


def row_sum_priority(A):
    rs = np.sum(A, axis=1)
    return normalize_positive(rs)


def arithmetic_mean_priority(A):
    # Same ranking as row_sum_priority, but included for completeness.
    am = np.mean(A, axis=1)
    return normalize_positive(am)


def column_sum_priority(A):
    col_sums = np.sum(A, axis=0)
    norm_matrix = A / col_sums
    w = np.sum(norm_matrix, axis=1)
    return normalize_positive(w)


def sscsm_priority(A):
    # Same ranking as column_sum_priority, but included for completeness.
    col_sums = np.sum(A, axis=0)
    w = np.sum(A / col_sums, axis=1)
    return normalize_positive(w)


def harmonic_mean_priority(A):
    n = A.shape[0]
    hm = n / np.sum(1.0 / A, axis=1)
    return normalize_positive(hm)


def cosine_maximization_priority(A):
    """
    Cosine maximisation method:
        max cosine(A, [w_i/w_j])

    Solved in bounded log-weight variables.
    """

    n = A.shape[0]
    norm_A = np.linalg.norm(A)

    if norm_A <= 0:
        raise ValueError("Invalid matrix norm.")

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        numerator = np.sum(A * R)
        norm_R = np.linalg.norm(R)

        if norm_R <= 0 or not np.isfinite(norm_R):
            return 1e100

        cosine = numerator / (norm_A * norm_R)

        if not np.isfinite(cosine):
            return 1e100

        return -cosine

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"CMM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def log_chebyshev_priority(A):
    """
    Log-Chebyshev method via linear programming:
        min max_ij |log(a_ij) - (x_i - x_j)|
    """

    n = A.shape[0]
    logA = np.log(A)

    num_vars = n + 1

    c = np.zeros(num_vars)
    c[-1] = 1.0

    bounds = [(None, None)] * n + [(0, None)]

    A_ub = []
    b_ub = []

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            # logA_ij - (x_i - x_j) <= t
            row1 = np.zeros(num_vars)
            row1[i] = -1.0
            row1[j] = 1.0
            row1[-1] = -1.0
            A_ub.append(row1)
            b_ub.append(-logA[i, j])

            # (x_i - x_j) - logA_ij <= t
            row2 = np.zeros(num_vars)
            row2[i] = 1.0
            row2[j] = -1.0
            row2[-1] = -1.0
            A_ub.append(row2)
            b_ub.append(logA[i, j])

    A_ub = np.asarray(A_ub)
    b_ub = np.asarray(b_ub)

    # Fix x_1 = 0 to remove translation invariance
    A_eq = np.zeros((1, num_vars))
    A_eq[0, 0] = 1.0
    b_eq = np.array([0.0])

    res = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not res.success:
        raise RuntimeError("Log-Chebyshev LP did not converge.")

    x = res.x[:n]
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def least_squares_priority(A):
    """
    Ordinary least squares method:
        min sum_ij (a_ij - w_i/w_j)^2
    """

    n = A.shape[0]

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        value = np.sum((A - R) ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"LSM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


def weighted_least_squares_priority(A):
    """
    Weighted least squares method:
        min sum_ij (a_ij w_j - w_i)^2
    """

    n = A.shape[0]

    z0 = geometric_mean_start(A)
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * n

    def objective(z):
        w = log_variables_to_weights(z)
        residual = A * w[None, :] - w[:, None]

        value = np.sum(residual ** 2)

        if not np.isfinite(value):
            return 1e100

        return value

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"WLSM optimisation did not converge: {res.message}")

    w = log_variables_to_weights(res.x)

    return normalize_positive(w)


def express_ahp_priority(A):
    """
    Reference-alternative / Express AHP style score.

    The reference alternative is chosen as the alternative with largest
    geometric mean score.
    """

    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    best = np.argmax(gm)

    # Scores relative to the selected reference alternative.
    w = A[:, best]

    return normalize_positive(w)


# =========================================================
# Ranking and comparison
# =========================================================

def ranking(weights):
    """
    Return ranking as zero-based indices.
    """
    return np.argsort(-weights)


def ranking_one_based(weights):
    """
    Return ranking as one-based indices, for paper-friendly output.
    """
    return ranking(weights) + 1


def pairwise_order_matrix(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def same_ranking(w1, w2, tol=RANK_TOL):
    return np.array_equal(
        pairwise_order_matrix(w1, tol),
        pairwise_order_matrix(w2, tol)
    )


# =========================================================
# Ordinal PCM generator
# =========================================================

def random_ordinal_pcm(n):
    """
    Generate random ordinal reciprocal matrix with entries:
    {1, P, 1/P}.
    """

    A = np.ones((n, n), dtype=object)

    for i in range(n):
        for j in range(i + 1, n):

            choice = np.random.choice(["tie", "pref", "notpref"])

            if choice == "tie":
                A[i, j] = 1
                A[j, i] = 1

            elif choice == "pref":
                A[i, j] = "P"
                A[j, i] = "1/P"

            else:
                A[i, j] = "1/P"
                A[j, i] = "P"

    return A


def substitute_alpha(A_ord, alpha):
    n = A_ord.shape[0]
    A = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(n):

            if A_ord[i, j] == "P":
                A[i, j] = alpha
            elif A_ord[i, j] == "1/P":
                A[i, j] = 1.0 / alpha
            else:
                A[i, j] = 1.0

    return A


# =========================================================
# Pretty printing
# =========================================================

def print_matrix(A):
    for row in A:
        print("   ".join(f"{x:7.3f}" for x in row))


def print_weights(w):
    print(", ".join(f"{x:.4f}" for x in w))


def print_latex_matrix_from_ordinal(A_ord):
    """
    Print a LaTeX matrix in terms of alpha.
    """

    n = A_ord.shape[0]

    print("\\[")
    print("A(\\alpha)=")
    print("\\begin{bmatrix}")

    for i in range(n):
        row = []

        for j in range(n):
            val = A_ord[i, j]

            if val == "P":
                row.append("\\alpha")
            elif val == "1/P":
                row.append("1/\\alpha")
            else:
                row.append("1")

        line = " & ".join(row)

        if i < n - 1:
            line += " \\\\"

        print(line)

    print("\\end{bmatrix}.")
    print("\\]")


# =========================================================
# Find and display explicit rank reversal
# =========================================================

def find_and_print_example(method, method_name, n=N_EXAMPLE, max_tries=MAX_TRIES):
    """
    Search for one explicit IOP rank reversal example.
    """

    for attempt in range(1, max_tries + 1):

        A_ord = random_ordinal_pcm(n)

        A_base = substitute_alpha(A_ord, BASELINE_ALPHA)
        w_base = method(A_base)

        for alpha in ALPHA_VALUES[1:]:

            A_alpha = substitute_alpha(A_ord, alpha)
            w_alpha = method(A_alpha)

            if not same_ranking(w_base, w_alpha):

                print("=" * 70)
                print(f"METHOD: {method_name}")
                print(f"Matrix size n = {n}")
                print(f"Attempt = {attempt}")
                print("-" * 70)

                print("\nSymbolic matrix:")
                print_latex_matrix_from_ordinal(A_ord)

                print(f"\nBaseline alpha = {BASELINE_ALPHA}")
                print(f"A({BASELINE_ALPHA}):")
                print_matrix(A_base)

                print("\nPriority vector:")
                print_weights(w_base)

                print("Ranking:", ranking_one_based(w_base))

                print(f"\nIntensified alpha = {alpha}")
                print(f"A({alpha}):")
                print_matrix(A_alpha)

                print("\nPriority vector:")
                print_weights(w_alpha)

                print("Ranking:", ranking_one_based(w_alpha))

                print("\n>>> IOP RANK REVERSAL DETECTED <<<")
                print("=" * 70 + "\n")

                return A_ord, BASELINE_ALPHA, alpha, w_base, w_alpha

    print(f"No example found for {method_name} after {max_tries} attempts.\n")
    return None


# =========================================================
# Main
# =========================================================

methods_for_examples = {
    # In the order of Section 2.2, excluding GMM because it is invariant.
    "Eigenvector method": eigenvector_priority,
    "Row sum method": row_sum_priority,
    "Column sum method": column_sum_priority,
    "Harmonic mean method": harmonic_mean_priority,
    "Cosine maximisation method": cosine_maximization_priority,
    "Log-Chebyshev method": log_chebyshev_priority,
    "Least squares method": least_squares_priority,
    "Weighted least squares method": weighted_least_squares_priority,

    # Optional additional methods from your earlier code:
    # "Arithmetic mean method": arithmetic_mean_priority,
    # "SSCSM": sscsm_priority,
    # "Express AHP": express_ahp_priority,
}

print("\nSearching explicit IOP rank reversal examples...\n")

examples_found = {}

for name, method in methods_for_examples.items():
    try:
        result = find_and_print_example(method, name, n=N_EXAMPLE, max_tries=MAX_TRIES)
        examples_found[name] = result
    except Exception as exc:
        print("=" * 70)
        print(f"METHOD: {name}")
        print(f"An error occurred: {exc}")
        print("=" * 70 + "\n")


# =========================================================
# Check geometric mean invariance
# =========================================================

print("\nChecking invariance of the geometric mean method...\n")

for test_id in range(5):
    A_ord = random_ordinal_pcm(N_EXAMPLE)

    A2 = substitute_alpha(A_ord, 2)
    A9 = substitute_alpha(A_ord, 9)

    w2 = geometric_mean_priority(A2)
    w9 = geometric_mean_priority(A9)

    print(f"Test {test_id + 1}")
    print("Ranking at alpha=2:", ranking_one_based(w2))
    print("Ranking at alpha=9:", ranking_one_based(w9))
    print("Same ranking:", same_ranking(w2, w9))
    print()

print("Geometric mean ranking is invariant under uniform preference intensification.\n")


Searching explicit IOP rank reversal examples...

METHOD: Eigenvector method
Matrix size n = 5
Attempt = 3
----------------------------------------------------------------------

Symbolic matrix:
\[
A(\alpha)=
\begin{bmatrix}
1 & 1 & 1 & \alpha & \alpha \\
1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1/\alpha & 1/\alpha \\
1/\alpha & 1 & \alpha & 1 & 1/\alpha \\
1/\alpha & 1 & \alpha & \alpha & 1
\end{bmatrix}.
\]

Baseline alpha = 2
A(2):
  1.000     1.000     1.000     2.000     2.000
  1.000     1.000     1.000     1.000     1.000
  1.000     1.000     1.000     0.500     0.500
  0.500     1.000     2.000     1.000     0.500
  0.500     1.000     2.000     2.000     1.000

Priority vector:
0.2636, 0.1888, 0.1514, 0.1713, 0.2249
Ranking: [1 5 2 4 3]

Intensified alpha = 9
A(9):
  1.000     1.000     1.000     9.000     9.000
  1.000     1.000     1.000     1.000     1.000
  1.000     1.000     1.000     0.111     0.111
  0.111     1.000     9.000     1.000     0.111
  0.111     1.000     9.000 

In [3]:
#Ejemplos donde cambia el primer término

In [2]:
# =========================================================
# Search for minimal top-ranked alternative change examples
# Retained methods only: EVM, RSM, CSM, HMM, CMM
# =========================================================

import os
import itertools
import numpy as np
import pandas as pd

from numpy.linalg import eig
from scipy.optimize import minimize


# =========================================================
# Configuration
# =========================================================

SEED = 42
rng = np.random.default_rng(SEED)

BASELINE_ALPHA = 2
ALPHA_VALUES = list(range(9, 2, -1))   # Try alpha = 9,8,...,3 first

RANK_TOL = 1e-10

N_MIN = 3
N_MAX = 9

# Exhaustive enumeration is feasible for small orders.
# For n=5, there are 3^10 = 59049 ordinal matrices.
EXHAUSTIVE_MAX_N = {
    "Eigenvector method": 5,
    "Row sum method": 5,
    "Column sum method": 5,
    "Harmonic mean method": 5,
    "Cosine maximisation method": 4,
}

# Random search limits for larger orders.
MAX_RANDOM_TRIES = {
    "Eigenvector method": 300000,
    "Row sum method": 2000000,
    "Column sum method": 500000,
    "Harmonic mean method": 1500000,
    "Cosine maximisation method": 500000,
}

PROGRESS_EVERY = 50000

LOG_WEIGHT_BOUND = 8.0
OPT_MAXITER = 300

OUTPUT_DIR = "minimal_top_rank_examples_retained_methods"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RI_VALUES = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}


# =========================================================
# Utility functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.any(~np.isfinite(w)):
        raise ValueError("Priority vector contains non-finite values.")

    if np.any(w < 0):
        raise ValueError("Priority vector contains negative values.")

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def geometric_mean_start(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    gm = normalize_positive(gm)
    return np.log(gm)


def fixed_scale_log_vector(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)
    return np.concatenate([z, np.array([0.0])])


def ratio_matrix_from_log_vector(x):
    x = np.asarray(x, dtype=float)
    d = x[:, None] - x[None, :]
    d = np.clip(d, -2 * LOG_WEIGHT_BOUND, 2 * LOG_WEIGHT_BOUND)
    return np.exp(d)


# =========================================================
# Priority derivation methods
# =========================================================

def principal_eigenpair(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[idx].real
    w = eigenvectors[:, idx].real

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    w = normalize_positive(w)

    return lambda_max, w


def eigenvector_priority(A):
    _, w = principal_eigenpair(A)
    return w


def geometric_mean_priority(A):
    gm = np.prod(A, axis=1) ** (1.0 / A.shape[0])
    return normalize_positive(gm)


def row_sum_priority(A):
    rs = np.sum(A, axis=1)
    return normalize_positive(rs)


def column_sum_priority(A):
    col_sums = np.sum(A, axis=0)
    norm_matrix = A / col_sums
    w = np.sum(norm_matrix, axis=1)
    return normalize_positive(w)


def harmonic_mean_priority(A):
    n = A.shape[0]
    hm = n / np.sum(1.0 / A, axis=1)
    return normalize_positive(hm)


def cosine_maximization_priority(A):
    n = A.shape[0]
    norm_A = np.linalg.norm(A)

    if norm_A <= 0:
        raise ValueError("Invalid matrix norm.")

    log_start = geometric_mean_start(A)
    z0 = log_start[:-1] - log_start[-1]
    z0 = np.clip(z0, -LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)

    bounds = [(-LOG_WEIGHT_BOUND, LOG_WEIGHT_BOUND)] * (n - 1)

    def objective(z):
        x = fixed_scale_log_vector(z)
        R = ratio_matrix_from_log_vector(x)

        numerator = np.sum(A * R)
        norm_R = np.linalg.norm(R)

        if norm_R <= 0 or not np.isfinite(norm_R):
            return 1e100

        cosine = numerator / (norm_A * norm_R)

        if not np.isfinite(cosine):
            return 1e100

        return -cosine

    res = minimize(
        objective,
        z0,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": OPT_MAXITER,
            "ftol": 1e-10,
        }
    )

    if not res.success:
        raise RuntimeError(f"CMM optimisation did not converge: {res.message}")

    x = fixed_scale_log_vector(res.x)
    w = np.exp(x - np.max(x))

    return normalize_positive(w)


# =========================================================
# Consistency ratio
# =========================================================

def consistency_ratio(A):
    n = A.shape[0]

    if n <= 2:
        return 0.0

    lambda_max, _ = principal_eigenpair(A)

    ci = (lambda_max - n) / (n - 1)
    ci = max(0.0, ci)

    ri = RI_VALUES.get(n)

    if ri is None:
        return np.nan

    if ri == 0:
        return 0.0

    return ci / ri


# =========================================================
# Ranking and comparison functions
# =========================================================

def ranking(weights):
    return np.argsort(-np.asarray(weights, dtype=float))


def ranking_one_based(weights):
    return ranking(weights) + 1


def pairwise_order_matrix(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def rank_reversal_occurred(w_base, w_new, tol=RANK_TOL):
    return not np.array_equal(
        pairwise_order_matrix(w_base, tol),
        pairwise_order_matrix(w_new, tol)
    )


def top_set(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    base_top = top_set(w_base, tol)
    new_top = top_set(w_new, tol)

    return len(base_top.intersection(new_top)) == 0


# =========================================================
# Ordinal PCM generation
# =========================================================

def ordinal_pcm_from_codes(n, codes):
    """
    Codes:
        0 = tie
        1 = i preferred to j
        2 = j preferred to i
    """

    A_ord = np.ones((n, n), dtype=object)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

    for code, (i, j) in zip(codes, pairs):

        if code == 0:
            A_ord[i, j] = 1
            A_ord[j, i] = 1

        elif code == 1:
            A_ord[i, j] = "P"
            A_ord[j, i] = "1/P"

        elif code == 2:
            A_ord[i, j] = "1/P"
            A_ord[j, i] = "P"

        else:
            raise ValueError("Invalid ordinal code.")

    return A_ord


def iter_ordinal_pcms_exhaustive(n):
    m = n * (n - 1) // 2

    for codes in itertools.product([0, 1, 2], repeat=m):
        yield ordinal_pcm_from_codes(n, codes)


def random_ordinal_pcm(n):
    m = n * (n - 1) // 2
    codes = rng.integers(0, 3, size=m)
    return ordinal_pcm_from_codes(n, codes)


def substitute_alpha(A_ord, alpha):
    n = A_ord.shape[0]
    A = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(n):

            if A_ord[i, j] == "P":
                A[i, j] = alpha
            elif A_ord[i, j] == "1/P":
                A[i, j] = 1.0 / alpha
            else:
                A[i, j] = 1.0

    return A


# =========================================================
# LaTeX formatting
# =========================================================

def latex_matrix_from_ordinal(A_ord):
    n = A_ord.shape[0]
    lines = []

    lines.append("\\[")
    lines.append("A(\\alpha)=")
    lines.append("\\begin{bmatrix}")

    for i in range(n):
        row = []

        for j in range(n):
            val = A_ord[i, j]

            if val == "P":
                row.append("\\alpha")
            elif val == "1/P":
                row.append("1/\\alpha")
            else:
                row.append("1")

        line = " & ".join(row)

        if i < n - 1:
            line += " \\\\"

        lines.append(line)

    lines.append("\\end{bmatrix}.")
    lines.append("\\]")

    return "\n".join(lines)


def latex_vector(w, decimals=3):
    return "(" + ",\\; ".join(f"{x:.{decimals}f}" for x in w) + ")"


def latex_ranking(rank):
    return " \\succ ".join(str(int(x)) for x in rank)


def print_matrix(A):
    for row in A:
        print("   ".join(f"{x:7.3f}" for x in row))


def print_weights(w):
    print(", ".join(f"{x:.4f}" for x in w))


# =========================================================
# Candidate evaluation
# =========================================================

def evaluate_candidate(A_ord, method, method_name, n, attempt, search_mode):
    try:
        A_base = substitute_alpha(A_ord, BASELINE_ALPHA)
        w_base = method(A_base)
    except Exception:
        return None

    for alpha in ALPHA_VALUES:

        try:
            A_alpha = substitute_alpha(A_ord, alpha)
            w_alpha = method(A_alpha)
        except Exception:
            continue

        rr = rank_reversal_occurred(w_base, w_alpha)
        top_change = top_rank_change_occurred(w_base, w_alpha)

        if top_change:

            top_base = top_set(w_base)
            top_alpha = top_set(w_alpha)

            result = {
                "method": method_name,
                "n": n,
                "attempt": attempt,
                "search_mode": search_mode,
                "baseline_alpha": BASELINE_ALPHA,
                "intensified_alpha": alpha,
                "cr_baseline_alpha_2": consistency_ratio(A_base),
                "A_ord": A_ord,
                "A_base": A_base,
                "A_alpha": A_alpha,
                "w_base": w_base,
                "w_alpha": w_alpha,
                "ranking_base": ranking_one_based(w_base),
                "ranking_alpha": ranking_one_based(w_alpha),
                "top_base": sorted([i + 1 for i in top_base]),
                "top_alpha": sorted([i + 1 for i in top_alpha]),
                "rank_reversal": rr,
                "top_change": top_change,
            }

            return result

    return None


# =========================================================
# Search functions
# =========================================================

def search_one_n_exhaustive(method, method_name, n):
    total_candidates = 3 ** (n * (n - 1) // 2)

    print(f"  Exhaustive search for n={n}; candidates={total_candidates}")

    for attempt, A_ord in enumerate(iter_ordinal_pcms_exhaustive(n), start=1):

        if attempt % PROGRESS_EVERY == 0:
            print(f"    {method_name}, n={n}: {attempt}/{total_candidates} checked")

        result = evaluate_candidate(
            A_ord=A_ord,
            method=method,
            method_name=method_name,
            n=n,
            attempt=attempt,
            search_mode="exhaustive",
        )

        if result is not None:
            return result, attempt, total_candidates

    return None, total_candidates, total_candidates


def search_one_n_random(method, method_name, n, max_tries):
    print(f"  Random search for n={n}; max_tries={max_tries}")

    for attempt in range(1, max_tries + 1):

        if attempt % PROGRESS_EVERY == 0:
            print(f"    {method_name}, n={n}: {attempt}/{max_tries} random matrices checked")

        A_ord = random_ordinal_pcm(n)

        result = evaluate_candidate(
            A_ord=A_ord,
            method=method,
            method_name=method_name,
            n=n,
            attempt=attempt,
            search_mode="random",
        )

        if result is not None:
            return result, attempt, max_tries

    return None, max_tries, max_tries


def minimality_status_for_result(result, search_log_method):
    smaller_orders = [
        row for row in search_log_method
        if row["n"] < result["n"]
    ]

    if all(row["mode"] == "exhaustive" and row["found"] is False for row in smaller_orders):
        return (
            "No example exists for smaller orders checked exhaustively; "
            "therefore the order is minimal within the enumerated ordinal class."
        )

    return (
        "Smallest order found by the sequential search. "
        "Some smaller orders were searched randomly, so absence there is not a proof."
    )


def find_minimal_example_for_method(method_name, method):
    search_log_method = []

    exhaustive_limit = EXHAUSTIVE_MAX_N.get(method_name, 4)
    random_limit = MAX_RANDOM_TRIES.get(method_name, 500000)

    for n in range(N_MIN, N_MAX + 1):

        print("-" * 70)
        print(f"{method_name}: searching order n={n}")

        if n <= exhaustive_limit:
            result, checked, planned = search_one_n_exhaustive(
                method=method,
                method_name=method_name,
                n=n,
            )
            mode = "exhaustive"

        else:
            result, checked, planned = search_one_n_random(
                method=method,
                method_name=method_name,
                n=n,
                max_tries=random_limit,
            )
            mode = "random"

        found = result is not None

        search_log_method.append({
            "method": method_name,
            "n": n,
            "mode": mode,
            "checked": checked,
            "planned": planned,
            "found": found,
        })

        if found:
            result["minimality_status"] = minimality_status_for_result(
                result,
                search_log_method,
            )
            return result, search_log_method

    return None, search_log_method


# =========================================================
# Display and save
# =========================================================

def display_example(result):
    if result is None:
        return

    print("=" * 70)
    print(f"METHOD: {result['method']}")
    print(f"Matrix order n = {result['n']}")
    print(f"Search mode = {result['search_mode']}")
    print(f"Attempt = {result['attempt']}")
    print(f"Baseline alpha = {result['baseline_alpha']}")
    print(f"Intensified alpha = {result['intensified_alpha']}")
    print(f"CR(A(2)) = {result['cr_baseline_alpha_2']:.4f}")
    print("-" * 70)

    print("\nSymbolic matrix:")
    print(latex_matrix_from_ordinal(result["A_ord"]))

    print(f"\nA({result['baseline_alpha']}):")
    print_matrix(result["A_base"])

    print("\nPriority vector at baseline:")
    print_weights(result["w_base"])

    print("Ranking at baseline:", result["ranking_base"])
    print("Top-ranked alternative(s) at baseline:", result["top_base"])

    print(f"\nA({result['intensified_alpha']}):")
    print_matrix(result["A_alpha"])

    print("\nPriority vector after intensification:")
    print_weights(result["w_alpha"])

    print("Ranking after intensification:", result["ranking_alpha"])
    print("Top-ranked alternative(s) after intensification:", result["top_alpha"])

    print("\nMinimality status:")
    print(result["minimality_status"])

    print("\n>>> TOP-RANKED ALTERNATIVE CHANGE DETECTED <<<")
    print("=" * 70 + "\n")


def save_example_as_latex(result):
    if result is None:
        return None

    safe_name = (
        result["method"]
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    file_path = os.path.join(
        OUTPUT_DIR,
        f"minimal_top_change_example_{safe_name}.tex"
    )

    alpha0 = result["baseline_alpha"]
    alpha1 = result["intensified_alpha"]

    text = []

    text.append(f"% Minimal top-ranked alternative change example for {result['method']}")
    text.append(f"% Matrix order: n={result['n']}")
    text.append(f"% Search mode: {result['search_mode']}")
    text.append(f"% Minimality status: {result['minimality_status']}")
    text.append("")
    text.append(latex_matrix_from_ordinal(result["A_ord"]))
    text.append("")
    text.append(
        f"For \\(\\alpha={alpha0}\\), the priority vector obtained by the "
        f"{result['method']} is"
    )
    text.append("\\[")
    text.append(f"w_{{A({alpha0})}}={latex_vector(result['w_base'])},")
    text.append("\\]")
    text.append("which induces the ranking")
    text.append("\\[")
    text.append(latex_ranking(result["ranking_base"]) + ".")
    text.append("\\]")
    text.append("")
    text.append(
        f"After uniform intensification to \\(\\alpha={alpha1}\\), "
        "the priority vector becomes"
    )
    text.append("\\[")
    text.append(f"w_{{A({alpha1})}}={latex_vector(result['w_alpha'])},")
    text.append("\\]")
    text.append("leading to")
    text.append("\\[")
    text.append(latex_ranking(result["ranking_alpha"]) + ".")
    text.append("\\]")
    text.append("")
    text.append(
        "Thus, the top-ranked alternative changes under uniform preference "
        "intensification, although the ordinal preference pattern is unchanged."
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("\n".join(text))

    print(f"LaTeX example saved to: {file_path}")

    return file_path


# =========================================================
# Retained methods only
# =========================================================

methods_for_top_examples = {
    "Eigenvector method": eigenvector_priority,
    "Row sum method": row_sum_priority,
    "Column sum method": column_sum_priority,
    "Harmonic mean method": harmonic_mean_priority,
    "Cosine maximisation method": cosine_maximization_priority,
}


# =========================================================
# Main search
# =========================================================

print("\nSearching minimal top-ranked alternative change examples...\n")

top_examples_found = {}
all_search_logs = []
summary_rows = []

for name, method in methods_for_top_examples.items():

    print("=" * 70)
    print(f"Searching method: {name}")
    print("=" * 70)

    result, search_log_method = find_minimal_example_for_method(
        method_name=name,
        method=method,
    )

    all_search_logs.extend(search_log_method)
    top_examples_found[name] = result

    if result is None:
        print(f"No top-ranked alternative change found for {name} up to n={N_MAX}.\n")

        summary_rows.append({
            "method": name,
            "found": False,
            "n": np.nan,
            "search_mode": "",
            "attempt": np.nan,
            "baseline_alpha": BASELINE_ALPHA,
            "intensified_alpha": np.nan,
            "cr_baseline_alpha_2": np.nan,
            "ranking_base": "",
            "ranking_alpha": "",
            "top_base": "",
            "top_alpha": "",
            "minimality_status": "No example found up to the maximum order searched.",
            "latex_file": "",
        })

    else:
        display_example(result)
        latex_file = save_example_as_latex(result)

        summary_rows.append({
            "method": name,
            "found": True,
            "n": result["n"],
            "search_mode": result["search_mode"],
            "attempt": result["attempt"],
            "baseline_alpha": result["baseline_alpha"],
            "intensified_alpha": result["intensified_alpha"],
            "cr_baseline_alpha_2": result["cr_baseline_alpha_2"],
            "ranking_base": " > ".join(map(str, result["ranking_base"])),
            "ranking_alpha": " > ".join(map(str, result["ranking_alpha"])),
            "top_base": ",".join(map(str, result["top_base"])),
            "top_alpha": ",".join(map(str, result["top_alpha"])),
            "minimality_status": result["minimality_status"],
            "latex_file": latex_file,
        })


# =========================================================
# Save summary files
# =========================================================

summary_df = pd.DataFrame(summary_rows)
search_log_df = pd.DataFrame(all_search_logs)

summary_path = os.path.join(
    OUTPUT_DIR,
    "minimal_top_change_examples_summary.csv"
)

search_log_path = os.path.join(
    OUTPUT_DIR,
    "minimal_top_change_examples_search_log.csv"
)

summary_df.to_csv(summary_path, index=False)
search_log_df.to_csv(search_log_path, index=False)

print("\nSummary saved to:")
print(summary_path)

print("\nSearch log saved to:")
print(search_log_path)

print("\nSummary:")
print(summary_df)


# =========================================================
# Sanity check: GMM invariance
# =========================================================

print("\nChecking geometric mean method invariance on random examples...\n")

for test_id in range(5):

    A_ord = random_ordinal_pcm(6)

    A2 = substitute_alpha(A_ord, 2)
    A9 = substitute_alpha(A_ord, 9)

    w2 = geometric_mean_priority(A2)
    w9 = geometric_mean_priority(A9)

    print(f"Test {test_id + 1}")
    print("Ranking at alpha=2:", ranking_one_based(w2))
    print("Ranking at alpha=9:", ranking_one_based(w9))
    print("Top set at alpha=2:", sorted([i + 1 for i in top_set(w2)]))
    print("Top set at alpha=9:", sorted([i + 1 for i in top_set(w9)]))
    print("Top-ranked alternative changes:", top_rank_change_occurred(w2, w9))
    print()

print("Search completed.")


Searching minimal top-ranked alternative change examples...

Searching method: Eigenvector method
----------------------------------------------------------------------
Eigenvector method: searching order n=3
  Exhaustive search for n=3; candidates=27
----------------------------------------------------------------------
Eigenvector method: searching order n=4
  Exhaustive search for n=4; candidates=729
----------------------------------------------------------------------
Eigenvector method: searching order n=5
  Exhaustive search for n=5; candidates=59049
METHOD: Eigenvector method
Matrix order n = 5
Search mode = exhaustive
Attempt = 792
Baseline alpha = 2
Intensified alpha = 9
CR(A(2)) = 0.0548
----------------------------------------------------------------------

Symbolic matrix:
\[
A(\alpha)=
\begin{bmatrix}
1 & 1 & 1 & 1 & \alpha \\
1 & 1 & 1 & 1 & 1/\alpha \\
1 & 1 & 1 & 1 & 1/\alpha \\
1 & 1 & 1 & 1 & 1/\alpha \\
1/\alpha & \alpha & \alpha & \alpha & 1
\end{bmatrix}.
\]

A(2

In [1]:
# =========================================================
# Search for an order-4 EVM IOP-rank reversal example
# with the lowest possible baseline CR(A(2))
# =========================================================

import itertools
import numpy as np
import pandas as pd
from numpy.linalg import eig

# =========================================================
# Configuration
# =========================================================

N = 4
BASELINE_ALPHA = 2
ALPHA_VALUES = list(range(3, 10))   # alpha = 3,...,9
RANK_TOL = 1e-10

RI_VALUES = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}

OUTPUT_CSV = "evm_order4_iop_rank_reversal_candidates.csv"
OUTPUT_TEX = "evm_order4_lowest_cr_example.tex"


# =========================================================
# Basic functions
# =========================================================

def normalize_positive(w):
    w = np.asarray(w, dtype=float)

    if np.sum(w) < 0:
        w = -w

    if np.any(w <= 0):
        w = np.abs(w)

    total = np.sum(w)

    if total <= 0:
        raise ValueError("Priority vector has non-positive sum.")

    return w / total


def principal_eigenpair(A):
    eigenvalues, eigenvectors = eig(A)
    idx = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[idx].real
    w = eigenvectors[:, idx].real
    w = normalize_positive(w)

    return lambda_max, w


def eigenvector_priority(A):
    _, w = principal_eigenpair(A)
    return w


def consistency_ratio(A):
    n = A.shape[0]

    if n <= 2:
        return 0.0

    lambda_max, _ = principal_eigenpair(A)

    ci = (lambda_max - n) / (n - 1)
    ci = max(0.0, ci)

    ri = RI_VALUES.get(n)

    if ri is None:
        return np.nan

    if ri == 0:
        return 0.0

    return ci / ri


def pairwise_order_matrix(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)
    n = len(w)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    order = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            diff = w[i] - w[j]

            if diff > eps:
                order[i, j] = 1
            elif diff < -eps:
                order[i, j] = -1
            else:
                order[i, j] = 0

    return order


def rank_reversal_occurred(w_base, w_new, tol=RANK_TOL):
    return not np.array_equal(
        pairwise_order_matrix(w_base, tol),
        pairwise_order_matrix(w_new, tol),
    )


def top_set(weights, tol=RANK_TOL):
    w = np.asarray(weights, dtype=float)

    scale = max(1.0, np.max(np.abs(w)))
    eps = tol * scale

    max_w = np.max(w)

    return set(np.where(max_w - w <= eps)[0])


def top_rank_change_occurred(w_base, w_new, tol=RANK_TOL):
    base_top = top_set(w_base, tol)
    new_top = top_set(w_new, tol)

    return len(base_top.intersection(new_top)) == 0


def ranking_one_based(weights):
    return list(np.argsort(-np.asarray(weights, dtype=float)) + 1)


# =========================================================
# Ordinal matrix construction
# =========================================================

def ordinal_pcm_from_codes(n, codes):
    """
    Codes:
        0 = tie
        1 = i preferred to j
        2 = j preferred to i

    Pairs are ordered as:
        (1,2), (1,3), ..., (1,n), (2,3), ..., (n-1,n)
    """

    A_ord = np.ones((n, n), dtype=object)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

    for code, (i, j) in zip(codes, pairs):

        if code == 0:
            A_ord[i, j] = 1
            A_ord[j, i] = 1

        elif code == 1:
            A_ord[i, j] = "P"
            A_ord[j, i] = "1/P"

        elif code == 2:
            A_ord[i, j] = "1/P"
            A_ord[j, i] = "P"

        else:
            raise ValueError("Invalid ordinal code.")

    return A_ord


def substitute_alpha(A_ord, alpha):
    n = A_ord.shape[0]
    A = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(n):

            if A_ord[i, j] == "P":
                A[i, j] = alpha
            elif A_ord[i, j] == "1/P":
                A[i, j] = 1.0 / alpha
            else:
                A[i, j] = 1.0

    return A


# =========================================================
# LaTeX formatting
# =========================================================

def latex_number_from_alpha_entry(x):
    if np.isclose(x, 1.0):
        return "1"

    if x > 1:
        if np.isclose(x, round(x)):
            return str(int(round(x)))
        return f"{x:.6g}"

    inv = 1.0 / x
    if np.isclose(inv, round(inv)):
        return f"1/{int(round(inv))}"

    return f"{x:.6g}"


def latex_matrix(A):
    lines = []
    lines.append("\\begin{bmatrix}")

    for i in range(A.shape[0]):
        row = [latex_number_from_alpha_entry(A[i, j]) for j in range(A.shape[1])]
        line = " & ".join(row)

        if i < A.shape[0] - 1:
            line += " \\\\"

        lines.append(line)

    lines.append("\\end{bmatrix}")

    return "\n".join(lines)


def latex_vector(w, decimals=3):
    return "(" + ",\\; ".join(f"{x:.{decimals}f}" for x in w) + ")"


def latex_ranking(rank):
    return " \\succ ".join(str(int(x)) for x in rank)


# =========================================================
# Exhaustive search
# =========================================================

def search_order4_evm_lowest_cr():
    n = N
    m = n * (n - 1) // 2

    candidates = []

    total = 3 ** m
    print(f"Exhaustive search for n={n}. Total candidates: {total}")

    for attempt, codes in enumerate(itertools.product([0, 1, 2], repeat=m), start=1):

        A_ord = ordinal_pcm_from_codes(n, codes)
        A_base = substitute_alpha(A_ord, BASELINE_ALPHA)

        try:
            w_base = eigenvector_priority(A_base)
            cr_base = consistency_ratio(A_base)
        except Exception:
            continue

        for alpha in ALPHA_VALUES:

            A_alpha = substitute_alpha(A_ord, alpha)

            try:
                w_alpha = eigenvector_priority(A_alpha)
            except Exception:
                continue

            rr = rank_reversal_occurred(w_base, w_alpha)
            top_change = top_rank_change_occurred(w_base, w_alpha)

            if rr:
                candidates.append({
                    "attempt": attempt,
                    "codes": codes,
                    "baseline_alpha": BASELINE_ALPHA,
                    "intensified_alpha": alpha,
                    "cr_A2": cr_base,
                    "top_rank_change": top_change,
                    "ranking_A2": " > ".join(map(str, ranking_one_based(w_base))),
                    "ranking_Aalpha": " > ".join(map(str, ranking_one_based(w_alpha))),
                    "w_A2": w_base,
                    "w_Aalpha": w_alpha,
                    "A_ord": A_ord,
                    "A2": A_base,
                    "Aalpha": A_alpha,
                })

                # Once a matrix has reversal, keep the first alpha that produces it.
                break

    if not candidates:
        raise RuntimeError("No order-4 EVM IOP-rank reversal example was found.")

    # Lowest CR first; if tied, use smaller alpha and then enumeration order.
    candidates = sorted(
        candidates,
        key=lambda d: (d["cr_A2"], d["intensified_alpha"], d["attempt"])
    )

    best = candidates[0]

    return best, candidates


best, candidates = search_order4_evm_lowest_cr()

# =========================================================
# Save complete candidate list
# =========================================================

summary_rows = []

for c in candidates:
    summary_rows.append({
        "attempt": c["attempt"],
        "codes": c["codes"],
        "baseline_alpha": c["baseline_alpha"],
        "intensified_alpha": c["intensified_alpha"],
        "cr_A2": c["cr_A2"],
        "top_rank_change": c["top_rank_change"],
        "ranking_A2": c["ranking_A2"],
        "ranking_Aalpha": c["ranking_Aalpha"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV, index=False)

# =========================================================
# Print best example
# =========================================================

print("\n" + "=" * 70)
print("BEST ORDER-4 EVM IOP-RANK REVERSAL EXAMPLE")
print("=" * 70)

print(f"Total reversal candidates found: {len(candidates)}")
print(f"Attempt: {best['attempt']}")
print(f"Codes: {best['codes']}")
print(f"Baseline alpha: {best['baseline_alpha']}")
print(f"Intensified alpha: {best['intensified_alpha']}")
print(f"CR(A(2)): {best['cr_A2']:.6f}")
print(f"Top-ranked alternative change: {best['top_rank_change']}")

print("\nA(2):")
print(best["A2"])

print("\nA(alpha):")
print(best["Aalpha"])

print("\nEVM weights at A(2):")
print(best["w_A2"])
print("Ranking at A(2):", best["ranking_A2"])

print("\nEVM weights after intensification:")
print(best["w_Aalpha"])
print("Ranking after intensification:", best["ranking_Aalpha"])

print("\nCandidate list saved to:")
print(OUTPUT_CSV)


# =========================================================
# LaTeX text for the paper
# =========================================================

alpha0 = best["baseline_alpha"]
alpha1 = best["intensified_alpha"]

# Since A(2) is the baseline and A(alpha1) = A(2)^(log(alpha1)/log(2)).
k_value = np.log(alpha1) / np.log(alpha0)

latex_text = f"""
% Order-4 EVM IOP-rank reversal example with lowest CR in the searched ordinal class.
% Exhaustive search over all 3^6 ordinal reciprocal matrices of order 4.
% Baseline alpha = {alpha0}; intensified alpha = {alpha1}.
% CR(A(2)) = {best['cr_A2']:.6f}.

Consider the reciprocal matrix
\\[
A=
{latex_matrix(best["A2"])}.
\\]
For this matrix, the consistency ratio is \\(\\mathrm{{CR}}(A)={best['cr_A2']:.4f}\\).
The eigenvector method yields
\\[
w_A={latex_vector(best["w_A2"])},
\\]
which induces the ranking
\\[
{latex_ranking(ranking_one_based(best["w_A2"]))}.
\\]
Now let \\(k=\\log({alpha1})/\\log({alpha0})={k_value:.3f}\\) and define the uniformly intensified matrix
\\(A^{{(k)}}=[a_{{ij}}^k]\\). The eigenvector method then gives
\\[
w_{{A^{{(k)}}}}={latex_vector(best["w_Aalpha"])},
\\]
leading to
\\[
{latex_ranking(ranking_one_based(best["w_Aalpha"]))}.
\\]
Hence, uniform preference intensification changes the induced ranking, although
the ordinal preference pattern is unchanged. This order-four example was selected
by exhaustive enumeration of the ordinal class with entries in \\(\\{{1,\\alpha,1/\\alpha\\}}\\)
as the one with the lowest baseline value of \\(\\mathrm{{CR}}(A(2))\\) among those
exhibiting EVM IOP-rank reversal.
""".strip()

with open(OUTPUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex_text)

print("\nLaTeX text saved to:")
print(OUTPUT_TEX)

print("\nLaTeX text:")
print(latex_text)

D:\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Exhaustive search for n=4. Total candidates: 729

BEST ORDER-4 EVM IOP-RANK REVERSAL EXAMPLE
Total reversal candidates found: 36
Attempt: 465
Codes: (1, 2, 2, 0, 1, 2)
Baseline alpha: 2
Intensified alpha: 4
CR(A(2)): 0.212268
Top-ranked alternative change: False

A(2):
[[1.  2.  0.5 0.5]
 [0.5 1.  1.  2. ]
 [2.  1.  1.  0.5]
 [2.  0.5 2.  1. ]]

A(alpha):
[[1.   4.   0.25 0.25]
 [0.25 1.   1.   4.  ]
 [4.   1.   1.   0.25]
 [4.   0.25 4.   1.  ]]

EVM weights at A(2):
[0.21781836 0.25813414 0.23465885 0.28938864]
Ranking at A(2): 4 > 2 > 3 > 1

EVM weights after intensification:
[0.21176224 0.26662831 0.21078353 0.31082593]
Ranking after intensification: 4 > 2 > 1 > 3

Candidate list saved to:
evm_order4_iop_rank_reversal_candidates.csv

LaTeX text saved to:
evm_order4_lowest_cr_example.tex

LaTeX text:
% Order-4 EVM IOP-rank reversal example with lowest CR in the searched ordinal class.
% Exhaustive search over all 3^6 ordinal reciprocal matrices of order 4.
% Baseline alpha = 2; inte